# ⚡ Ultra-Fast WebGPU Chrome on Google Colab (Tesla T4) — 60 FPS WebRTC
این نوت‌بوک محیط لینوکس ابری Google Colab را با کارت گرافیک **NVIDIA Tesla T4** و شتاب سخت‌افزاری **WebGPU** راه‌اندازی کرده و از طریق **Google Chrome Remote Desktop (WebRTC ۶۰ فریم و تاخیر ناچیز زیر ۳۰ میلی‌ثانیه)** استریم می‌کند.

### 🌟 چرا Chrome Remote Desktop (CRD) به جای VNC معمولی؟
* **تاخیر نزدیک به صفر (Sub-30ms):** استفاده مستقیم از پروتکل WebRTC و شبکه اینترنت داخلی دیتاسنترهای گوگل (بسیار سریع‌تر از هر تونل واسط).
* **۶۰ فریم بر ثانیه واقعی و پایدار:** استفاده از کدک‌های ویدیویی مدرن (VP8/VP9) به جای تکه‌تکه فرستادن تصاویر (Screen Scraping در VNC قدیمی).
* **نشانگر موس محلی (Zero Input Lag):** موس شما بدون کوچک‌ترین تاخیر یا پرش حرکت می‌کند و حس کار با کامپیوتر شخصی را دارد.
* **کلیپ‌بورد دوطرفه و رزولوشن تطبیق‌پذیر:** کپی-پیست مستقیم و سازگاری خودکار با اندازه صفحه نمایش شما.

---
### 📋 مراحل راه‌اندازی (فقط ۲ دقیقه):
1. مطمئن شوید نوع Runtime روی **T4 GPU** است: (`Runtime` ➔ `Change runtime type` ➔ `T4 GPU`).
2. سلول **[1] نصب درایورهای گرافیک و Chrome Remote Desktop** را اجرا کنید (~۴۵ ثانیه).
3. در تب دیگر به آدرس **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)** بروید، دکمه‌های **Begin ➔ Next ➔ Authorize** را بزنید و کدی که برای سیستم Debian Linux می‌دهد را کپی کنید.
4. کد را در فیلد `AUTH_COMMAND` سلول **[2]** پیست کرده و سلول را اجرا کنید.
5. وارد **[remotedesktop.google.com/access](https://remotedesktop.google.com/access)** شوید؛ دستگاه `colab-t4` ظاهر شده و با وارد کردن پین وارد دسکتاپ فوق‌العاده روان ۶۰ فریم می‌شوید!


In [ ]:
#@title ⚡ [1/2] پیکربندی سخت‌افزاری تسلا T4، درایورهای Vulkan و پکیج‌های ریموت دسکتاپ
#@markdown اجرای این سلول درایورها، کرنل‌نودهای GPU، دسکتاپ XFCE4، گوگل کروم استیبل و سرور ریموت را آماده می‌کند (~۴۵ ثانیه).

import os, sys, time, subprocess

print("⏳ [1/5] پیکربندی درایورهای سطح هسته NVIDIA Tesla T4 و Vulkan...")
os.makedirs("/dev/dri", exist_ok=True)
subprocess.run("mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null", shell=True)
subprocess.run("mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null", shell=True)
subprocess.run("mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null", shell=True)
subprocess.run("chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null", shell=True)

os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
os.makedirs("/usr/share/vulkan/icd.d", exist_ok=True)
icd_content = '{\n    "file_format_version": "1.0.0",\n    "ICD": {\n        "library_path": "libGLX_nvidia.so.0",\n        "api_version": "1.3.275"\n    }\n}'
for p in ["/etc/vulkan/icd.d/nvidia_icd.json", "/usr/share/vulkan/icd.d/nvidia_icd.json"]:
    with open(p, "w") as f: f.write(icd_content)

with open("/etc/ld.so.conf.d/nvidia.conf", "w") as f:
    f.write("/usr/lib64-nvidia\n")
subprocess.run("ldconfig", shell=True)

print("⏳ [2/5] ایجاد کاربر غیراز روت و تنظیم مجوزهای دسترسی به GPU...")
subprocess.run("id -u colab &>/dev/null || useradd -m -s /bin/bash colab", shell=True)
subprocess.run("echo 'colab:123456' | chpasswd", shell=True)
subprocess.run("usermod -aG sudo,video,render colab 2>/dev/null", shell=True)
with open("/etc/sudoers.d/colab", "w") as f:
    f.write("colab ALL=(ALL) NOPASSWD:ALL\n")

print("⏳ [3/5] دانلود و نصب Google Chrome Remote Desktop...")
crd_deb = "/tmp/crd.deb"
subprocess.run(f"wget -q -O {crd_deb} https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb", shell=True)
subprocess.run(f"dpkg --install {crd_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
subprocess.run(f"rm -f {crd_deb}", shell=True)
subprocess.run("usermod -aG chrome-remote-desktop colab 2>/dev/null", shell=True)

print("⏳ [4/5] نصب محیط گرافیکی سبک XFCE4...")
subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal xscreensaver > /dev/null 2>&1", shell=True)
subprocess.run("systemctl disable lightdm.service > /dev/null 2>&1 || true", shell=True)

crd_session = "exec /etc/X11/Xsession /usr/bin/xfce4-session\n"
with open("/etc/chrome-remote-desktop-session", "w") as f:
    f.write(crd_session)
with open("/home/colab/.chrome-remote-desktop-session", "w") as f:
    f.write(crd_session)
subprocess.run("chown colab:colab /home/colab/.chrome-remote-desktop-session && chmod +x /home/colab/.chrome-remote-desktop-session", shell=True)

print("⏳ [5/5] بررسی و راه‌اندازی Google Chrome Stable با شتاب کامل سخت‌افزاری...")
if not os.path.exists("/usr/bin/google-chrome"):
    chrome_deb = "/tmp/chrome.deb"
    subprocess.run(f"wget -q -O {chrome_deb} https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb", shell=True)
    subprocess.run(f"dpkg -i {chrome_deb} > /dev/null 2>&1 || apt-get install -fy -qq > /dev/null 2>&1", shell=True)
    subprocess.run(f"rm -f {chrome_deb}", shell=True)

launch_script = """#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-vulkan=native \\
  --use-angle=vulkan \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --disable-background-timer-throttling \\
  --disable-backgrounding-occluded-windows \\
  --disable-renderer-backgrounding \\
  --start-maximized \\
  "$@" \\
  "https://unicred.fun/#mine" &
"""
with open("/home/colab/launch_chrome.sh", "w") as f:
    f.write(launch_script)
subprocess.run("chmod +x /home/colab/launch_chrome.sh", shell=True)

os.makedirs("/home/colab/.config/autostart", exist_ok=True)
os.makedirs("/home/colab/Desktop", exist_ok=True)
desktop_entry = """[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Comment=Google Chrome with Hardware WebGPU on Tesla T4
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Path=/home/colab
Terminal=false
StartupNotify=true
"""
with open("/home/colab/.config/autostart/webgpu-chrome.desktop", "w") as f:
    f.write(desktop_entry)
with open("/home/colab/Desktop/WebGPU_Chrome.desktop", "w") as f:
    f.write(desktop_entry)
subprocess.run("chmod +x /home/colab/Desktop/WebGPU_Chrome.desktop", shell=True)
subprocess.run("chown -R colab:colab /home/colab", shell=True)

print("\n" + "═"*60)
print("✅ مرحله اول انجام شد! همه پیش‌نیازها با موفقیت نصب شدند.")
print("👉 اکنون کد را از https://remotedesktop.google.com/headless کپی کرده و در سلول بعدی قرار دهید.")
print("═"*60)


In [ ]:
#@title 🔑 [2/2] راه‌اندازی هاست Chrome Remote Desktop (استریم ۶۰ فریم WebRTC)
#@markdown ### راهنمای دریافت دستور احراز هویت:
#@markdown 1. در یک تب جدید، به صفحه **[Chrome Remote Desktop Headless](https://remotedesktop.google.com/headless)** بروید.
#@markdown 2. دکمه **Begin** و سپس **Next** و سپس **Authorize** را بزنید.
#@markdown 3. دستوری که برای سیستم لینوکس دبیان به شما نمایش می‌دهد (که با `DISPLAY= /opt/google/...` شروع می‌شود) را کپی کنید.
#@markdown 4. دستور را در فیلد **AUTH_COMMAND** زیر قرار دهید و دکمه **Play** را بزنید.

AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://unicred.fun/#mine" #@param {type:"string"}

import os, sys, time, subprocess, re
from IPython.display import display, HTML

if not AUTH_COMMAND.strip():
    print("❌ لطفاً دستور احراز هویت را در فیلد AUTH_COMMAND بالا پیست کنید!")
    print("👈 لینک دریافت آسان کد: https://remotedesktop.google.com/headless")
else:
    # بروزرسانی URL هدف در صورت تغییر
    if os.path.exists("/home/colab/launch_chrome.sh"):
        with open("/home/colab/launch_chrome.sh", "r") as f:
            c = f.read()
        c = re.sub(r'"https://[^"]+"', f'"{TARGET_URL}"', c)
        with open("/home/colab/launch_chrome.sh", "w") as f:
            f.write(c)

    raw_cmd = AUTH_COMMAND.strip()
    
    # پشتیبانی از فرمت‌های مختلف (فقط کد، یا دستور کامل)
    if raw_cmd.startswith("4/"):
        exec_cmd = f'/opt/google/chrome-remote-desktop/start-host --code="{raw_cmd}" --redirect-url="https://remotedesktop.google.com/_/oauthredirect" --name=colab-t4'
    else:
        exec_cmd = raw_cmd
    
    pin_str = str(PIN)
    if "--pin=" not in exec_cmd:
        exec_cmd += f" --pin={pin_str}"
    
    exec_cmd = re.sub(r'^DISPLAY=\s*', '', exec_cmd)
    
    print("⏳ در حال راه‌اندازی هاست ریموت دسکتاپ گوگل با اکانت شما...")
    
    full_cmd = f'su - colab -c \'{exec_cmd}\''
    proc = subprocess.Popen(
        full_cmd,
        shell=True,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    try:
        stdout, stderr = proc.communicate(input=f"{pin_str}\n{pin_str}\n", timeout=25)
    except subprocess.TimeoutExpired:
        proc.kill()
        stdout, stderr = proc.communicate()

    time.sleep(3)
    check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop", shell=True, capture_output=True, text=True)
    
    print("\n" + "═"*60)
    if "chrome-remote-desktop" in check.stdout:
        print("🎉 فوق‌العاده است! سرویس Chrome Remote Desktop با موفقیت راه‌اندازی شد.")
        print(f"🔑 پین ورود شما: {pin_str}")
        print("═"*60)
        display(HTML(f'''
            <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
                <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری آماده اتصال است! (WebRTC 60 FPS)</h2>
                <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                    سیستم شما با نام <b>colab-t4</b> در پنل گوگل اضافه شد. روی دکمه زیر کلیک کنید و پین <b>{pin_str}</b> را وارد نمایید:
                </p>
                <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                    👉 ورود به Chrome Remote Desktop
                </a>
                <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                    آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
                </p>
            </div>
        '''))
    else:
        print("⚠️ پیام خروجی:")
        print(stdout)
        if stderr:
            print("خطاها:", stderr)
        print("═"*60)
        print("نکته: اگر خطای OAuth یا منقضی شدن کد دریافت کردید، کد قبلی را ببندید و یک کد جدید از remotedesktop.google.com/headless تولید کنید.")


In [ ]:
#@title 🌐 [گزینه جانبی] راه‌اندازی سریع با مرورگر و تونل Cloudflare (بدون نیاز به اکانت گوگل)
#@markdown اگر ترجیح می‌دهید بدون ورود به اکانت گوگل مستقیماً در همین تب مرورگر تست کنید، می‌توانید این سلول را اجرا نمایید:

TARGET_URL = "https://unicred.fun/#mine" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1600x900", "1920x1080"]

import os, sys, time, subprocess, re
from IPython.display import display, HTML

print("⏳ راه‌اندازی استک مرورگر در تب وب...")
subprocess.run("apt-get update -qq && apt-get install -y -qq xvfb fluxbox x11vnc websockify novnc autocutsel > /dev/null 2>&1", shell=True)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True)

subprocess.run("pkill -9 -f 'Xvfb|x11vnc|websockify|fluxbox|cloudflared'", shell=True)
time.sleep(1)

env = os.environ.copy()
env["DISPLAY"] = ":1"
env["VK_ICD_FILENAMES"] = "/etc/vulkan/icd.d/nvidia_icd.json"

subprocess.Popen(["Xvfb", ":1", "-screen", "0", f"{RESOLUTION}x24"], env=env)
time.sleep(1)
subprocess.Popen(["fluxbox"], env=env)
subprocess.Popen(["autocutsel", "-fork", "-s", "CLIPBOARD"], env=env)
subprocess.Popen(["x11vnc", "-display", ":1", "-nopw", "-listen", "127.0.0.1", "-forever", "-rfbport", "5900", "-wait", "5", "-defer", "5", "-threads"], env=env)
subprocess.Popen(["websockify", "--web", "/usr/share/novnc", "6080", "localhost:5900"], env=env)
time.sleep(1)

subprocess.Popen([
    "google-chrome",
    "--no-sandbox",
    "--disable-gpu-sandbox",
    "--enable-unsafe-webgpu",
    "--enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService",
    "--use-vulkan=native",
    "--use-angle=vulkan",
    "--ignore-gpu-blocklist",
    "--disable-dev-shm-usage",
    "--enable-gpu-rasterization",
    "--enable-zero-copy",
    "--start-maximized",
    "--user-data-dir=/tmp/chrome_colab_profile",
    TARGET_URL
], env=env)

log_file = "/tmp/cloudflared.log"
if os.path.exists(log_file): os.remove(log_file)
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:6080", "--logfile", log_file])

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file) as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    link = f"{tunnel_url}/vnc.html?autoconnect=true&resize=scale&quality=6&compression=2"
    display(HTML(f'<br><a href="{link}" target="_blank" style="padding:12px 24px; background:#43a047; color:#fff; font-weight:bold; font-size:16px; text-decoration:none; border-radius:6px; display:inline-block;">👉 باز کردن در مرورگر (noVNC)</a><br><br>لینک مستقیم: {link}'))


In [ ]:
#@title 📊 مانیتور زنده هش‌ریت و مصرف گرافیک تسلا T4 (Real-Time GPU Monitor)
#@markdown اجرای این سلول وضعیت زنده درصد درگیری GPU، هش‌ریت ماینینگ، توان مصرفی و دمای کارت گرافیک را نمایش می‌دهد.

import time, subprocess
from IPython.display import clear_output

print("Starting lightweight GPU monitor (press Stop button in notebook to halt)...")
try:
    while True:
        res = subprocess.run([
            "nvidia-smi",
            "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw",
            "--format=csv,noheader,nounits"
        ], capture_output=True, text=True)
        parts = [p.strip() for p in res.stdout.strip().split(",")]
        if len(parts) >= 7:
            name, gpu_util, mem_util, mem_used, mem_total, temp, power = parts
            clear_output(wait=True)
            print("="*60)
            print(f"🚀 UNICRED WEBGPU MINING MONITOR — {name}")
            print("="*60)
            print(f"  GPU Compute Utilization : {gpu_util}%")
            print(f"  VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
            print(f"  GPU Temperature         : {temp} °C")
            print(f"  Power Consumption       : {power} W")
            print("="*60)
        time.sleep(1)
except KeyboardInterrupt:
    print("Monitor stopped.")
